In [ ]:
from synthesizer import distr_concat, load_json_to_df
import json
import pandas as pd
from scipy.spatial import KDTree
from sklearn.neighbors import NearestNeighbors
import numpy as np
import audioflux as af
import analysis

In [64]:
with open("metadata\metro_sample_2.json", "r") as f:
    df = pd.read_json(f)
df = df.fillna(0)
cols_to_check = df.iloc[:, 3:] #remove all the "silent" rows/grains
df = df[cols_to_check.ne(0).any(axis=1)]
vector_df = df.iloc[:, 3:]
normalized_df=(vector_df-vector_df.min())/(vector_df.max()-vector_df.min())
tree = KDTree(normalized_df)
normalized_df

,centroid,flatness,kurtosis,flux,energy,crest,rms,eef,eer,band_width,decrease,entropy,spread,slope
20,0.549345,0.674666,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.984834,0.000000,0.342871,0.999999
21,1.000000,1.000000,0.013909,7.467325e-10,5.764326e-10,0.089204,0.000025,0.144578,0.654132,0.006142,0.965712,1.000000,0.709941,1.000000
22,0.670746,0.846890,0.015085,2.718034e-09,1.184461e-09,0.304155,0.000030,0.144578,0.654132,0.005431,0.856800,0.901836,0.810096,0.999973
23,0.770870,0.933253,0.010082,3.309464e-09,1.720362e-09,0.172081,0.000043,0.144578,0.654132,0.008399,0.934554,0.971510,0.757671,0.999970
24,0.409753,0.377061,0.059493,3.600651e-06,3.500907e-06,0.313580,0.001871,0.144601,0.654139,0.038211,0.920732,0.793428,0.550259,0.998030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9368,0.055416,0.026733,0.482343,2.171198e-02,1.426306e-02,0.590961,0.131170,0.201078,0.683308,0.104330,0.644149,0.554965,0.062171,0.846791
9369,0.068781,0.029482,0.427966,2.341775e-02,1.235580e-02,0.524126,0.126434,0.194392,0.678607,0.106869,0.803712,0.580153,0.073415,0.854732
9370,0.069897,0.031795,0.454414,2.633765e-02,1.438808e-02,0.477165,0.129239,0.199865,0.681753,0.107532,0.745806,0.581834,0.070716,0.846796
9371,0.078969,0.030814,0.409462,1.684907e-02,9.676584e-03,0.422131,0.115143,0.187740,0.673115,0.102931,0.852217,0.603713,0.068320,0.860775


In [63]:
seed = np.random.seed(111)
random_starting_point = np.random.rand(1, len(vector_df.iloc[0]))[0]
nn = tree.query(random_starting_point)
random_starting_point, nn

(array([0.61217018, 0.16906975, 0.43605902, 0.76926247, 0.2953253 ,
        0.14916296, 0.02247832, 0.42022449, 0.23868214, 0.33765619,
        0.99071246, 0.23772645, 0.08119266, 0.66960024]),
 (1.110117533097052, np.int64(3049)))

In [125]:
# generate path across n-dimensional space
start = normalized_df.iloc[0]
stop = normalized_df.iloc[-1]
# start, end
trajectory = np.linspace(start,stop,num=10) #default 50 points: 10 second is 10*100 if sr is 48kHz
num_particles = 20
n_features = len(vector_df.iloc[0])
particles = trajectory[0] + np.random.normal(0, 0.1, (num_particles, n_features)) #change 0.1 for spread degree
k = 5
weights = np.array([np.ones(k) / k for _ in range(num_particles)] )

output_indices = []
total_steps = len(trajectory)
sigma = 0.1 #the smaller the value, the smaller the distance between the nearest neighbour and target grain
# sigma needs to be very high, when alpha is high
alpha = 0.5

y, sr = af.read("corpus\metro_sample_2.wav", samplate=48000)
if y.ndim > 1:
    y = np.mean(y, axis=1)

output_buffer = np.array([])

for t in range(total_steps):
    target = trajectory[t]
    particles += np.random.normal(0, alpha, particles.shape) #add noise to nearby particles 
    iteration = []
    for i in range(num_particles):
        dist, grain_idxs = tree.query(particles[i], k=k) 
        for _ in range(len(grain_idxs)):  
            grain_idx = np.random.choice(grain_idxs)  
            row = df.iloc[grain_idx]
            grain_start = int(row["grain_start"])
            grain_end = grain_start + int(row["grain_size"])
            grain = y[grain_start: grain_end]
            output_buffer = np.concatenate([output_buffer, grain])
        # weights[i] = np.exp(-dist**2 / (2 * sigma**2)) #Gaussian radial basis func, dist >> prob
        # print(weights[i])
        # break
    # weights /= np.sum(weights)
    # # print(weights.max)

    # best_particle_idx = np.argmax(weights)
    # _, final_grain_idx = tree.query(particles[best_particle_idx])
    # # print("INDEX: ", final_grain_idx, normalized_df.iloc[final_grain_idx])    
    # # break
    # output_indices.append(final_grain_idx)
    
    # indices = np.random.choice(range(num_particles), size=num_particles, p=weights)
    # particles = particles[indices]
    # print(weights)
af.write(data=output_buffer, path="output/nn_concat_1.wav", samplate=sr)

In [12]:
with open("metadata\metro_sample_2.json", "r") as f:
    df = pd.read_json(f)
df = df.fillna(0)
cols_to_check = df.iloc[:, 3:] #remove all the "silent" rows/grains
df = df[cols_to_check.ne(0).any(axis=1)]
vector_df = df.iloc[:, 7:10]
# vector_df
normalized_df=(vector_df-vector_df.min())/(vector_df.max()-vector_df.min())
tree = KDTree(normalized_df)
# normalized_df

In [163]:
# generate path across n-dimensional space
start = normalized_df.iloc[0]
stop = normalized_df.iloc[-1]
# start, end
trajectory = np.linspace(start,stop,num=10) #default 50 points: 10 second is 10*100 if sr is 48kHz
num_particles = 20
n_features = len(vector_df.iloc[0])
particles = trajectory[0] + np.random.normal(0, 0.1, (num_particles, n_features)) #change 0.1 for spread degree
k = 2
weights = np.array([np.ones(k) / k for _ in range(num_particles)] )

output_indices = []
total_steps = len(trajectory)
sigma = 0.1 #the smaller the value, the smaller the distance between the nearest neighbour and target grain
# sigma needs to be very high, when alpha is high
alpha = 0.5

y, sr = af.read("corpus\metro_sample_2.wav", samplate=48000)
if y.ndim > 1:
    y = np.mean(y, axis=1)

output_buffer = np.array([])

for t in range(total_steps):
    target = trajectory[t]
    particles += np.random.normal(0, alpha, particles.shape) #add noise to nearby particles 
    iteration = []
    for i in range(num_particles):
        dist, grain_idxs = tree.query(particles[i], k=k) 
        for _ in range(len(grain_idxs)):  
            grain_idx = np.random.choice(grain_idxs)  
            row = df.iloc[grain_idx]
            grain_start = int(row["grain_start"])
            grain_end = grain_start + int(row["grain_size"])
            grain = y[grain_start: grain_end]
            grain = grain * np.bartlett(len(grain))
            output_buffer = np.concatenate([output_buffer, grain])
        # weights[i] = np.exp(-dist**2 / (2 * sigma**2)) #Gaussian radial basis func, dist >> prob
        # print(weights[i])
        # break
    # weights /= np.sum(weights)
    # # print(weights.max)

    # best_particle_idx = np.argmax(weights)
    # _, final_grain_idx = tree.query(particles[best_particle_idx])
    # # print("INDEX: ", final_grain_idx, normalized_df.iloc[final_grain_idx])    
    # # break
    # output_indices.append(final_grain_idx)
    
    # indices = np.random.choice(range(num_particles), size=num_particles, p=weights)
    # particles = particles[indices]
    # print(weights)
af.write(data=output_buffer, path="output/nn_concat_6.wav", samplate=sr)

In [1]:
import analysis
onset_slicer = analysis.OnsetSlicer(
    # source="corpus\metro_sample_2.wav",
    # indices="testtest.csv"
)
onset_slicer.run(
    source="corpus\metro_sample_2.wav",
    indices="testtest.csv"
)


'  8%\n 26%\n 47%\n 66%\n 85%\n 99%\n100%\n'

In [13]:
import numpy as np
import sounddevice as sd

In [14]:
df.sort_values(by="energy")

y, sr = af.read("corpus\metro_sample_2.wav", samplate=48000)
y = np.mean(y, axis=1) if y.ndim > 1 else y
s = int(df.iloc[0]["grain_start"])
print(s)

grain= y[s: s+4800] * np.hanning(len(y[s: s+4800]))

sd.play(grain, samplerate=sr)

9600


In [28]:
import csv
with open("testtest.csv", "r") as f:
    indices = np.loadtxt(f, delimiter=",")

# for index in indices:
# print(int(index))
# print(indices[0])
i = int(indices[0])
grain = y[i: i+4800]
print(grain)
print(f"playing grain with index: {i}")
sd.play(grain,sr)
sd.wait()


[-0.02767944 -0.0128479  -0.03042603 ... -0.04144287 -0.04467773
 -0.00027466]
playing grain with index: 19456


In [43]:
i = int(indices[50])
grain = y[i: i+4800]
grain = grain * np.hanning(len(grain))
print(grain)
print(f"playing grain with index: {i}")
sd.play(grain,sr)
# sd.wait()

[ 0.00000000e+00 -1.24242978e-08 -2.30385618e-07 ... -2.44300825e-08
  1.07372090e-08  0.00000000e+00]
playing grain with index: 207360


In [44]:
np.zeros(2) + np.array([1,2])

array([1., 2.])

In [ ]:
density = 5 #20Hz
grain_size = 48000//2
# sr is 48kHz
input_length = len(y)
duration = 20 #in seconds
output_length = sr*duration #5 seconds
output_buffer = np.zeros(output_length)

np.random.seed(11111)
sampled = []
for _ in range(5):# stochastic 20 grains
    grain_index = int(np.random.choice(indices))
    while grain_index in sampled:
        grain_index = int(np.random.choice(indices))
    # print(grain_index)
    sampled.append(grain_index)
    grain_end = grain_index+grain_size if not grain_index+grain_size > input_length else input_length
    
    for i in range(duration):
        for j in range(density):
            start, end = j*(sr//density), j*(sr//density)+grain_size
            start, end = start + i*sr, end+i*sr
            if end > output_length:
                end = output_length

            grain = y[grain_index: grain_end] * np.hamming(grain_end-grain_index) 
            # if len(grain) != (end-start):
            #     print(len(grain), (end-start))
            # break
            grain = grain[:int(np.min([end-start, len(grain)]))]
            # if len(grain) != (end-start):
            #     print(len(grain), (end-start))
            if np.random.uniform() > 0.5:
                try:
                    output_buffer[start: end] += grain
                except Exception as e:
                    grain = grain [:len(output_buffer[start: end])]
                    # print(len(grain), print(len(output_buffer[start: end])))
                    output_buffer[start: end] += grain

output_buffer /= np.max(np.abs(output_buffer))
af.write(path="output/onset_gs_7.wav", data=output_buffer,samplate=sr)


ValueError: operands could not be broadcast together with shapes (24000,) (1024,) (24000,) 

In [100]:
np.min([1,2])

np.int64(1)

In [80]:
s,e = sampled[0], sampled[0]+4800

sd.play(y[s:e], samplerate=48000)